# **Data Preprocessing & Outlier Handling**
Dataset: Google Play Store Apps

**Importing Required Libraries**

Explanation:
We import all necessary libraries for data manipulation, cleaning, and visualization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import pandas as pd
import os
import re


pd.set_option('display.max_columns', None)
sns.set(style='whitegrid', palette='crest')


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Load the dataset**

We import the CSV file and load it into a pandas DataFrame.

In [2]:

path = kagglehub.dataset_download("meenakshsinghania/googleplaystore")

print("Path to dataset files:", path)

csv_path = None
for file in os.listdir(path):
    if file.endswith(".csv"):
        csv_path = os.path.join(path, file)
        break

if csv_path is None:
    raise FileNotFoundError(f"No .csv file found in {path}")

df = pd.read_csv(csv_path)
df.head()



100%|██████████| 312k/312k [00:01<00:00, 295kB/s]

Extracting files...
Path to dataset files: /Users/meenakshsinghania04/.cache/kagglehub/datasets/meenakshsinghania/googleplaystore/versions/1


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


**Dataset Overview**

Explanation:
Get basic details about data shape, column types, and missing values.

In [3]:
print("Dataset Shape:", df.shape)
df.info()
df.isna().sum()




Dataset Shape: (10841, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

**Remove invalid rating values**

Any rows where the app rating is above 5 are removed because such values are not possible.

In [4]:
df = df[(df["Rating"].isna()) | (df["Rating"] <= 5)]

**Create helper functions**

Small helper functions are created to clean installs, price, size, and version columns. These functions fix formatting issues and convert messy text into proper numeric or standardized values.

In [5]:
def clean_installs(x):
    if pd.isna(x): return np.nan
    return pd.to_numeric(str(x).replace(",", "").replace("+", "").strip(), errors="coerce")

def clean_price(x):
    if pd.isna(x): return np.nan
    return pd.to_numeric(str(x).replace("$", "").strip(), errors="coerce")

def clean_size(x):
    if pd.isna(x): return np.nan
    t = str(x).strip().replace(" ", "").lower()
    if t in ["varieswithdevice", "nan", "none", ""]: return np.nan
    m = re.match(r"([0-9]*\.?[0-9]+)([kmg]?)", t)
    if not m: return np.nan
    num, unit = float(m.group(1)), m.group(2)
    if unit == "k": return num / 1024
    if unit == "m": return num
    if unit == "g": return num * 1024
    return num

def clean_android_ver(x):
    if pd.isna(x) or "varies" in str(x).lower(): return np.nan
    m = re.search(r"(\d+(\.\d+){0,2})", str(x))
    return m.group(1) if m else np.nan

def clean_current_ver(x):
    if pd.isna(x) or "varies" in str(x).lower(): return np.nan
    v = re.sub(r"[^0-9\.]", "", str(x).lower().strip())
    return v if (v != "" and v.count(".") <= 3) else np.nan

def cap_outliers(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    return np.clip(s, Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

**Clean important columns**

ReviewsAll important columns are cleaned and standardized. Reviews and Installs are converted into clean integers. Price values are converted into numeric dollars. App Size is converted into MB and missing sizes are filled with the median. “Last Updated” is converted into a date and the year is extracted. Android Version and Current Version are cleaned to extract valid version numbers. The Genres column is simplified by keeping only the primary genre.

In [6]:
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce").fillna(0)
df.loc[df["Reviews"] % 1 != 0, "Reviews"] = np.floor(df["Reviews"])
df["Reviews"] = df["Reviews"].astype("Int64")

df["Installs"] = df["Installs"].apply(clean_installs).fillna(0).astype("Int64")

df["Price"] = df["Price"].apply(clean_price).fillna(0)

df["Size"] = df["Size"].apply(clean_size)

df.rename(columns={"Size": "Size_MB"}, inplace=True)
df["Size_MB"] = df.groupby("Category")["Size_MB"].transform(
    lambda x: x.fillna(x.median())
).round(2)

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

df["Rating"] = df.groupby("Category")["Rating"].transform(
    lambda x: x.fillna(x.median())
)



df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df["Last_Updated_Year"] = df["Last Updated"].dt.year

df["Android Ver"] = df["Android Ver"].apply(clean_android_ver)
df["Current Ver"] = df["Current Ver"].apply(clean_current_ver)

df["Primary_Genre"] = df["Genres"].apply(lambda x: str(x).split(";")[0])

**Fill Missing Categories**

Columns like Content Rating, Category, and Primary Genre have missing values replaced using the most common value (mode).

In [7]:
for col in ["Content Rating", "Primary_Genre", "Category"]:
    df[col] = df[col].fillna(df[col].mode()[0])

**Fix Version Missing Values**

Blank or invalid version fields are treated as missing and replaced with the label “Not Provided” for consistency.

In [8]:
df["Android Ver"] = df["Android Ver"].replace(r'^\s*$', np.nan, regex=True)
df["Current Ver"] = df["Current Ver"].replace(r'^\s*$', np.nan, regex=True)

df = df.dropna(subset=["Android Ver", "Current Ver"]).reset_index(drop=True)


**Create App Type**

A new column is created that labels apps as Free or Paid based on whether the price is zero or greater than zero.

In [9]:
df["Type"] = np.where(df["Price"] > 0, "Paid", "Free")


**Rename Price Column**

The “Price” column is renamed to Price_$ to clearly show it represents the cost in dollars.

In [10]:
df = df.rename(columns={"Price": "Price_$"})

**Remove Duplicate Apps**

If the same app appears multiple times, the version with the highest number of reviews is kept because it is the most reliable record.

In [11]:
df = df.loc[df.groupby("App")["Reviews"].idxmax()].reset_index(drop=True)


**Cap Outliers**

Extreme values in numeric columns like Reviews, Installs, Size, and Rating are capped using the IQR method to prevent them from affecting analysis.

In [12]:
num_outliers = ["Reviews", "Installs", "Size_MB", "Rating"]
df[num_outliers] = df[num_outliers].astype(float).apply(cap_outliers)
df["Reviews"] = df["Reviews"].round().astype("Int64")
df["Installs"] = df["Installs"].round().astype("Int64")

**Save Cleaned EDA Dataset**

A fully cleaned dataset is saved as googleplaystore_clean_EDA.csv, which is used for exploratory data analysis.

In [13]:
df.to_csv("googleplaystore_cleaned.csv", index=False)
print("Saved cleaned dataset as 'googleplaystore_cleaned.csv'")


Saved cleaned dataset as 'googleplaystore_cleaned.csv'
